# EXAMS (mhardalov/exams) : data inspection
NB: IF ON EDSTEM THEY SAY IT IS ALLOWED TO MERGE TRAIN/VAL/TEST AND RE-SPLIT TO TRAIN/VAL, USE THE TEST SET HERE!

In [14]:
from datasets import load_dataset
import pandas as pd
import re
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

TARGET_LANGUAGES = {'Italian', 'Spanish'}
ds = load_dataset("mhardalov/exams", "multilingual")
print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answerKey', 'info'],
        num_rows: 7961
    })
    validation: Dataset({
        features: ['id', 'question', 'answerKey', 'info'],
        num_rows: 2672
    })
    test: Dataset({
        features: ['id', 'question', 'answerKey', 'info'],
        num_rows: 13510
    })
})
{'id': '35dd69ff-7e71-11ea-9eb1-54bef70b159e', 'question': {'stem': 'Редуцентите са организми, които се хранят:', 'choices': {'text': ['само с животинска храна', 'с мъртва органична материя', 'чрез фотосинтеза', 'само с растителна храна'], 'label': ['A', 'B', 'C', 'D'], 'para': ['', '', '', '']}}, 'answerKey': 'B', 'info': {'grade': 12, 'subject': 'Biology', 'language': 'Bulgarian'}}


In [12]:
rows = []

for split in ["train", "validation", "test"]:
    for ex in ds[split]:
        info = ex["info"]
        q = ex["question"]
        choices = q["choices"]

        rows.append({
            "split": split,
            "id": ex["id"],
            "language": info.get("language"),
            "subject": info.get("subject"),
            "grade": info.get("grade"),
            "stem": q.get("stem"),
            "labels": choices.get("label"),
            "texts": choices.get("text"),
            "paras": choices.get("para"),
            "answerKey": ex.get("answerKey"),
            "n_choices": len(choices.get("label", [])),
        })

df = pd.DataFrame(rows)

In [15]:
df.head()

,split,id,language,subject,grade,stem,labels,texts,paras,answerKey,n_choices
0,train,35dd69ff-7e71-11ea-9eb1-54bef70b159e,Bulgarian,Biology,12,"Редуцентите са организми, които се хранят:","[A, B, C, D]","[само с животинска храна, с мъртва органична материя, чрез фотосинтеза, само с растителна храна]","[, , , ]",B,4
1,train,35dd6a01-7e71-11ea-9eb1-54bef70b159e,Bulgarian,Biology,12,"Кои химични съединения, характерни за клетката, задължително съдържат в молекулата си, както aминогрупа (–NH2), така също и карбоксилна група (–COOH)?","[A, B, C, D]","[аминокиселини, мастни киселини, фосфолипиди, монозахариди]","[, , , ]",A,4
2,train,35dd6a06-7e71-11ea-9eb1-54bef70b159e,Bulgarian,Biology,12,Кои от изброените структури са характерни САМО за растителните клетки?,"[A, B, C, D]","[клетъчна стена и пластиди, апарат на Голджи и ендоплазмена мрежа, митохондрии и рибозоми, клетъчен център и лизозоми]","[, , , ]",A,4
3,train,35dd6a08-7e71-11ea-9eb1-54bef70b159e,Bulgarian,Biology,12,"Хлоропластите и митохондриите си приличат по това, че в тях:","[A, B, C, D]","[протича цикъла на Калвин, протича цикъла на Кребс, няма ензими, се синтезира АТФ]","[, , , ]",D,4
4,train,35dd6a13-7e71-11ea-9eb1-54bef70b159e,Bulgarian,Biology,12,Гъстата космена покривка при бозайниците през зимата е пример за:,"[A, B, C, D]","[генетична адаптация, морфологична адаптация, физиологична адаптация, поведенческа адаптация]","[, , , ]",B,4


In [8]:
# Language distribution
print(df['language'].value_counts().to_string())

language
Bulgarian           2937
Croatian            2879
Vietnamese          2443
Hungarian           2267
North Macedonian    2075
Polish              1971
Turkish             1964
Serbian             1637
Albanian            1505
Italian             1256
Portuguese           924
Lithuanian           593
German               577
Arabic               562
French               318
Spanish              235


In [10]:
# Filter to target languages
df_target = df[df['language'].isin(TARGET_LANGUAGES)].copy()
print(f'rows for Italian/Spanish: {len(df_target)}')
print(df_target['language'].value_counts())

rows for Italian/Spanish: 1491
language
Italian    1256
Spanish     235
Name: count, dtype: int64


In [ ]:
# Choice count distribution — may be 3, 4 or 5
print('Choice count distribution:')
print(df_target.groupby(['language', 'n_choices']).size().to_string())

In [17]:
# check answerKey values
print('Answer key values:')
print(df_target['answerKey'].value_counts())

# flag answer keys not in labels
bad_key = df_target[df_target.apply(
    lambda r: r['answerKey'] not in r['labels'], axis=1
)]
print(f'\nRows where answer_key not in labels: {len(bad_key)}')
if len(bad_key):
    display(bad_key[['id', 'language', 'stem', 'labels', 'answer_key']].head(10))

Answer key values:
answerKey
B    410
C    396
A    386
D    298
E      1
Name: count, dtype: int64

Rows where answer_key not in labels: 0


In [18]:
# check support paragraphs (para field) — are they ever non-empty?
def has_para(paras):
    return any(p and str(p).strip() for p in paras)

df_target['has_para'] = df_target['paras'].apply(has_para)
print(f'Rows with non-empty support paragraph: {df_target["has_para"].sum()}')
print(df_target.groupby(['language', 'has_para']).size())

# Show an example with a para
with_para = df_target[df_target['has_para']]
if len(with_para):
    ex = with_para.iloc[0]
    print(f'\nExample with para:')
    print(f'  stem: {ex["stem"][:100]}')
    print(f'  paras: {ex["paras"]}')

Rows with non-empty support paragraph: 0
language  has_para
Italian   False       1256
Spanish   False        235
dtype: int64


In [19]:
# encoding corruption check
corruption = re.compile(r'[\ufffd\x00-\x08\x0b\x0c\x0e-\x1f]')

def has_corruption(row):
    fields = [row['stem']] + list(row['texts'])
    return any(corruption.search(str(f)) for f in fields)

df_target['has_corruption'] = df_target.apply(has_corruption, axis=1)
print(f'Rows with encoding corruption: {df_target["has_corruption"].sum()}')
if df_target['has_corruption'].any():
    display(df_target[df_target['has_corruption']][['language', 'stem', 'texts']].head(5))

Rows with encoding corruption: 0


In [20]:
# subject distribution per language
for lang in TARGET_LANGUAGES:
    sub = df_target[df_target['language'] == lang]
    print(f'\n=== {lang} ({len(sub)} rows) ===')
    print(sub['subject'].value_counts().to_string())


=== Italian (1256 rows) ===
subject
Biology        238
History        228
Chemistry      137
Physics        133
Sociology      110
Psychology     106
Politics       105
Informatics     97
Philosophy      38
Ethics          25
Geology         24
Geography       15

=== Spanish (235 rows) ===
subject
Physics      211
Geography     24


In [37]:
# spot-check formatted examples
for lang in TARGET_LANGUAGES:
    sub = df_target[df_target['language'] == lang]
    row = sub.iloc[0]
    print(f'=== {lang} ===')
    print(f'Q: {row["stem"]}')
    for label, text in zip(row['labels'], row['texts']):
        print(f'  {label}) {text}')
    print(f'Answer: {row["answerKey"]}')
    print()

=== Italian ===
Q: Attraverso l’osservazione di un organismo si è visto che le sue cellule contengono il nucleo, i mitocondri ed i cloroplasti. Ai livelli di organizzazione superiori si è costatato che non sono presenti né tessuti né organi. A quale degli organismi elencati di seguito corrisponde tale descrizione?
  A) il politrico
  B) il paramecio
  C) la verongia
  D) l’euglena
Answer: D

=== Spanish ===
Q: ¿Cuál de las siguientes velocidades es la menor?
  A) 1 m/s
  B) 7,2 km/h.
  C) 0,0036 km/s.
Answer: A



In [43]:
# Cleaning decision: NEED TO WAIT FOR EDSTEM ANSWER IF WE CAN MERGE ALL SPLITS AND RESPLIT TO TRAIN/VAL?
def valid_row(r):
    labels = r["labels"]
    texts = r["texts"]

    return (
        r["split"] in {"train", "validation"}
        and r["answerKey"] in labels
        and not r["has_corruption"]
        and str(r["stem"]).strip() != ""
        and isinstance(labels, list)
        and isinstance(texts, list)
        and len(labels) == len(texts)
        and 2 <= len(labels)
        and all(str(x).strip() != "" for x in texts)
    )

clean = df_target[df_target.apply(valid_row, axis=1)].copy()
rejected = df_target[~df_target.index.isin(clean.index)].copy()

print(f"Clean:    {len(clean)}")
print(f"Rejected: {len(rejected)}")
print(clean.groupby(["split", "language", "n_choices"]).size())

Clean:    620
Rejected: 871
split       language  n_choices
train       Italian   3             43
                      4            421
validation  Italian   3             14
                      4            142
dtype: int64


In [48]:
# Save cleaned version
import json
from pathlib import Path

MULTILINGUAL_DIR = Path("/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual")
out_dir = MULTILINGUAL_DIR / "data" / "processed" / "exams"
out_dir.mkdir(parents=True, exist_ok=True)

# Convert list columns to JSON strings for CSV compatibility
for col in ['labels', 'texts', 'paras']:
    clean[col] = clean[col].apply(json.dumps)
    rejected[col] = rejected[col].apply(json.dumps)

clean.to_csv(out_dir / 'exams_clean_all.csv', index=False)
rejected.to_csv(out_dir / 'exams_rejected_all.csv', index=False)

for lang in TARGET_LANGUAGES:
    lang_clean = clean[clean['language'] == lang]
    lang_clean.to_csv(out_dir / f'exams_clean_{lang.lower()}.csv', index=False)
    print(f'Saved {lang}: {len(lang_clean)} rows')

print(f'Saved clean:    {out_dir / "exams_clean_all.csv"}')
print(f'Saved rejected: {out_dir / "exams_rejected_all.csv"}')

Saved Italian: 620 rows
Saved Spanish: 0 rows
Saved clean:    /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/exams/exams_clean_all.csv
Saved rejected: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/exams/exams_rejected_all.csv


In [47]:
# Summary
print('=== SUMMARY ===')
print(f'Total multilingual rows:         {len(df)}')
print(f'Italian + Spanish rows:          {len(df_target)}')
print(f'Clean after filtering:           {len(clean)}')
print(f'Rejected:                        {len(rejected)}')
print('\nChoice count in clean data:')
print(clean.groupby(['language', 'n_choices']).size().to_string())

=== SUMMARY ===
Total multilingual rows:         24143
Italian + Spanish rows:          1491
Clean after filtering:           620
Rejected:                        871

Choice count in clean data:
language  n_choices
Italian   3             57
          4            563
